In [ ]:
# ==========================================
# 🛠️ STEP 1: INSTALL TOOLS (Run Every New Session)
# ==========================================

print(" Installing libraries.")
!pip install -q -U langchain-huggingface langchain-community langchain-core transformers accelerate bitsandbytes

print(" Installation Complete! Now can import the modules.")

 Installing libraries.
 Installation Complete! Now can import the modules.


In [ ]:
# # ==========================================
# # 🛠️ THE "DIRECT CONNECTOR"
# # ==========================================

import torch
from langchain_core.language_models.llms import LLM
from typing import Any, List, Optional
from langchain_core.callbacks.manager import CallbackManagerForLLMRun
from transformers import AutoTokenizer, AutoModelForCausalLM
from google.colab import drive
import os
drive.mount('/content/drive')
# # 1. Mount Google Drive
if not os.path.exists('/content/drive'):
    print("🔌 Mounting Google Drive...")

# 1. Define a Custom Adapter (This is the Magic Fix)
class QwenCustomLLM(LLM):
    model: Any
    tokenizer: Any

    def _call(
        self,
        prompt: str,
        stop: Optional[List[str]] = None,
        run_manager: Optional[CallbackManagerForLLMRun] = None,
        **kwargs: Any,
    ) -> str:
        # A. Manual Tokenization (We do this ourselves to avoid the error)
        inputs = self.tokenizer(prompt, return_tensors="pt").to(self.model.device)

        # B. Manual Generation (Directly telling the brain to think)
        with torch.no_grad():
            outputs = self.model.generate(
                **inputs,
                max_new_tokens=512,
                temperature=0.7,
                do_sample=True,
                pad_token_id=self.tokenizer.pad_token_id
            )

        # C. Decode (Convert math back to text)
        # We slice [inputs.shape[1]:] to remove the Question from the Answer
        generated_text = self.tokenizer.decode(outputs[0][inputs.input_ids.shape[1]:], skip_special_tokens=True)
        return generated_text

    #@property
    def _llm_type(self) -> str:
        return "qwen_custom"

# 2. Load the Model (Standard)
print("🚀 Loading Qwen directly...")
MODEL_PATH = "/content/drive/MyDrive/My_Local_AI/Models/Qwen2.5-1.5B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_PATH,
    torch_dtype=torch.float16,
    device_map="auto",
    low_cpu_mem_usage=True
)

# 3. Activate the Custom Adapter
print("\n🔗 Activating Direct Connection...")
llm = QwenCustomLLM(model=model, tokenizer=tokenizer)

# 4. Test it!
print("\n🧪 Testing...")
try:
    response = llm.invoke("Hi Qwen! Are you working now?")
    print(f"✅ IT WORKS! Response:\n{response}")
except Exception as e:
    print(f"❌ Still failing: {e}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
!pip install -q langchain langchain-community pypdf sentence-transformers faiss-cpu xgboost scikit-learn nltk

In [ ]:
from langchain_community.document_loaders import PyPDFLoader
import os

BOOK_DIR = "/content/drive/MyDrive/books"

documents = []

for file in os.listdir(BOOK_DIR):
    if file.endswith(".pdf"):
        loader = PyPDFLoader(os.path.join(BOOK_DIR, file))
        docs = loader.load()
        documents.extend(docs)

print("Total pages loaded:", len(documents))


Total pages loaded: 638


In [ ]:
if documents:
    print("First document's page content:\n", documents[0].page_content)
    print("\nFirst document's metadata:\n", documents[0].metadata)
else:
    print("No documents were loaded.")

First document's page content:
 

First document's metadata:
 {'producer': 'calibre 3.27.1 [https://calibre-ebook.com]', 'creator': 'calibre 3.27.1 [https://calibre-ebook.com]', 'creationdate': '2018-10-18T05:47:09+00:00', 'author': 'James Clear', 'title': 'Atomic Habits: Tiny Changes, Remarkable Results', 'source': '/content/drive/MyDrive/books/Atomic Habits by James Clear.pdf.pdf', 'total_pages': 285, 'page': 0, 'page_label': '1'}


In [ ]:
!pip install -q langchain-experimental

In [ ]:
from langchain_experimental.text_splitter import SemanticChunker
from langchain_huggingface import HuggingFaceEmbeddings

embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

semantic_splitter = SemanticChunker(
    embeddings,
    breakpoint_threshold_type="percentile",
    breakpoint_threshold_amount=95
)

chunks = semantic_splitter.split_documents(documents)

print("Total semantic chunks:", len(chunks))

Total semantic chunks: 1564


In [ ]:
from langchain_community.vectorstores import FAISS

vectorstore = FAISS.from_documents(chunks, embeddings)
vectorstore.save_local("semantic_faiss_db")

print("FAISS vector DB built successfully")

FAISS vector DB built successfully


In [ ]:
def semantic_retrieve(query, k=10):
    return vectorstore.similarity_search(query, k=k)

In [ ]:
docs = semantic_retrieve("how to build discipline and consistency")

docs[:2]

[Document(id='a2582681-6deb-41ce-8235-70abb736a83a', metadata={'producer': 'calibre 3.27.1 [https://calibre-ebook.com]', 'creator': 'calibre 3.27.1 [https://calibre-ebook.com]', 'creationdate': '2018-10-18T05:47:09+00:00', 'author': 'James Clear', 'title': 'Atomic Habits: Tiny Changes, Remarkable Results', 'source': '/content/drive/MyDrive/books/Atomic Habits by James Clear.pdf.pdf', 'total_pages': 285, 'page': 190, 'page_label': '191'}, page_content='MASTERING\tA\tFIELD\nFIGURE\t16:\t\nThe\tprocess\tof\tmastery\trequires\tthat\tyou\tprogressively\tlayer\nimprovements\ton\ttop\tof\tone\tanother,\teach\thabit\tbuilding\tupon\tthe\tlast\tuntil\ta\nnew\tlevel\tof\tperformance\thas\tbeen\treached\tand\ta\thigher\trange\tof\tskills\thas\nbeen\tinternalized. Although\thabits\tare\tpowerful,\twhat\tyou\tneed\tis\ta\tway\tto\tremain\nconscious\tof\tyour\tperformance\tover\ttime,\tso\tyou\tcan\tcontinue\tto\t\nrefine\nand\timprove.'),
 Document(id='8dca5501-59be-498d-8242-3d908bf424d3', metadat

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

vectorizer = TfidfVectorizer(ngram_range=(1,2))

def extract_features(query, docs):
    texts = [doc.page_content for doc in docs]
    corpus = [query] + texts

    tfidf = vectorizer.fit_transform(corpus)

    q_vec = tfidf[0]
    doc_vecs = tfidf[1:]

    tfidf_sim = cosine_similarity(q_vec, doc_vecs)[0]

    keyword_overlap = [
        len(set(query.lower().split()) & set(text.lower().split()))
        for text in texts
    ]

    chunk_lengths = [len(text.split()) for text in texts]

    return np.column_stack([
        tfidf_sim,
        keyword_overlap,
        chunk_lengths
    ])

In [ ]:
docs = semantic_retrieve("how to stay disciplined every day")
X = extract_features("how to stay disciplined every day", docs)

X.shape, X[:3]

((10, 3),
 array([[1.19516756e-01, 3.00000000e+00, 1.08000000e+02],
        [6.37273025e-03, 1.00000000e+00, 1.16000000e+02],
        [4.45709897e-03, 1.00000000e+00, 9.30000000e+01]]))

In [ ]:
def generate_training_data(samples=3000, k=10):
    X, y = [], []

    for _ in range(samples):

        idx = np.random.randint(len(chunks))
        text = chunks[idx].page_content

        if len(text) < 200:
            continue

        query = text[:200]

        docs = semantic_retrieve(query, k)
        features = extract_features(query, docs)

        relevance = (
            0.6 * np.random.rand(len(docs)) +
            0.3 * features[:,0] +
            0.1 * features[:,1]
        )

        X.extend(features)
        y.extend(relevance)

    return np.array(X), np.array(y)

In [ ]:
X, y = generate_training_data(samples=3000)
print("Training Data Shape:", X.shape, y.shape)

Training Data Shape: (24940, 3) (24940,)


In [ ]:
from xgboost import XGBRegressor

ranker = XGBRegressor(
    n_estimators=300,
    max_depth=6,
    learning_rate=0.05,
    subsample=0.9,
    colsample_bytree=0.9,
    objective='reg:squarederror'
)

ranker.fit(X, y)

XGBRegressor(base_score=None, booster=None, callbacks=None,
             colsample_bylevel=None, colsample_bynode=None,
             colsample_bytree=0.9, device=None, early_stopping_rounds=None,
             enable_categorical=False, eval_metric=None, feature_types=None,
             feature_weights=None, gamma=None, grow_policy=None,
             importance_type=None, interaction_constraints=None,
             learning_rate=0.05, max_bin=None, max_cat_threshold=None,
             max_cat_to_onehot=None, max_delta_step=None, max_depth=6,
             max_leaves=None, min_child_weight=None, missing=nan,
             monotone_constraints=None, multi_strategy=None, n_estimators=300,
             n_jobs=None, num_parallel_tree=None, ...)

In [ ]:
def hybrid_retrieve(query, k=10, final_k=3):

    docs = semantic_retrieve(query, k)

    features = extract_features(query, docs)
    scores = ranker.predict(features)

    ranked = sorted(zip(docs, scores), key=lambda x: x[1], reverse=True)

    return ranked[:final_k]

In [ ]:
results = hybrid_retrieve("how to stay disciplined when motivation is low")

for i, (doc, score) in enumerate(results, 1):
    print(f"\n🔹 Rank {i} | Score: {score:.4f}\n")
    print(doc.page_content[:500])


🔹 Rank 1 | Score: 0.8392

We	think	we	act	better	than	we	do. Measurement	offers	one	way	to	overcome	our	blindness	to	our	own
behavior	and	notice	what’s	really	going	on	each	day. One	glance	at	the
paper	clips	in	the	container	and	you	immediately	know	how	much
work	you	have	(or	haven’t)	been	putting	in. When	the	evidence	is	right
in	front	of	you,	you’re	less	likely	to	lie	to	yourself. Benefit	#2:	Habit	tracking	is	attractive. The	most	effective	form	of	motivation	is	progress. When	we	get	a
signal	that	we	are	moving	forward

🔹 Rank 2 | Score: 0.7959

Once	a	habit	has	been	established,	however,	it’s	important	to
continue	to	advance	in	small	ways. These	little	improvements	and	new
challenges	keep	you	engaged. And	if	you	hit	the	Goldilocks	Zone	just
right,	you	can	achieve	a	
flow	state
. *
A	flow	state	is	the	experience	of	being	“in	the	zone”	and	fully
immersed	in	an	activity. Scientists	have	tried	to	quantify	this	feeling. They	found	that	to	achieve	a	state	of	flow,	a	task	must	be	roughly	

In [ ]:
results = hybrid_retrieve("i sleep at 5 am and wake up at 11 am what are the chances of my sucess")

for i, (doc, score) in enumerate(results, 1):
    print(f"\n🔹 Rank {i} | Score: {score:.4f}\n")
    print(doc.page_content[:600])


🔹 Rank 1 | Score: 1.4565

I	hope
that’s	not	a	dumb	question. Is	that	too	basic?”
“It’s	a	great	question,”	said	the	artist	as	he	stroked	his	girlfriend’s	back. “It’s	a	fabulous	question!”	exclaimed	the	tycoon. “And	sure. Like	I’ve
suggested,	buy	an	old-school	alarm	clock—that’s	what	I	use. As	I	said	in
Agra,	you	never	want	to	sleep	with	any	technology	in	your	bedroom. I’ll
explain	why	soon. Once	you	have	your	alarm	clock,	advance	the	time	on	it
from	the	actual	time	to	thirty	minutes	ahead. Then	set	the	alarm	for	5:30	
AM
.”
“Really?”	indicated	the	artist. “That	seems	odd.”
“I	know,”	admitted	the	billionaire. “But	it	wor

🔹 Rank 2 | Score: 1.3924

I’ll	restructure	my	pre-
sleep	routine	so	that	I	wake	up	at	5	
AM
	feeling	better	and	full	of	energy. I
promise	 to	 do	 this	 so	 I	 rest	 well	 and	 so	 I	 can	 implement	
The	 20/20/20
Formula
	flawlessly.”

🔹 Rank 3 | Score: 1.3344

Let’s	 rock	 this	 piece. Please	 know	 that	 your	 creativity,
productivity,	prosperity,	performance	and	us